# 1. Imports

In [2]:
import pandas as pd
import numpy as np
import glob
import polars as pl
from pathlib import Path

DATA_PATH = Path("./data/UpdatedData/citibike_2023_combined.parquet/")

# 2. Data Load & Process

In [3]:
df = (
    pl.read_parquet(f"{DATA_PATH}/*.parquet")
    # 1) Parse to datetime and rename in one go
    .with_columns(
        pl.col("started_at").str.to_datetime().alias("start_time"),
        pl.col("ended_at").str.to_datetime().alias("end_time"),
    )
    # 2) Compute ride_time_seconds = end_time - start_time (in seconds)
    .with_columns(
        (pl.col("end_time") - pl.col("start_time"))
        .dt.total_seconds()
        .alias("ride_time_seconds")
    )
    # 3) Order by start_time
    .sort("start_time")
    # 4) Select columns in the exact order you want
    .select(
        "ride_id",
        "rideable_type",
        "start_time",
        "end_time",
        "ride_time_seconds",   # ← right after end_time
        "start_station_name",
        "start_station_id",
        "end_station_name",
        "end_station_id",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng",
        "member_casual",
    )
)

df.head()


ride_id,rideable_type,start_time,end_time,ride_time_seconds,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
str,str,datetime[μs],datetime[μs],i64,str,str,str,str,f64,f64,f64,f64,str
"""1371D252BF823F8F""","""classic_bike""",2022-12-14 11:43:54.762,2023-01-08 14:08:53.507,2168698,"""Dock St & Front St""","""4903.09""",null,null,40.702709,-73.99253,null,null,"""casual"""
"""49762460E71B1340""","""classic_bike""",2022-12-28 09:17:28.882,2023-01-01 09:24:57.541,346048,"""Flushing Ave & Vanderbilt Ave""","""4762.05""","""Washington Ave & Park Ave""","""4724.03""",40.69795,-73.970776,40.696102,-73.96751,"""casual"""
"""F04D6905667F3109""","""classic_bike""",2022-12-28 09:17:51.549,2023-01-01 07:39:38.879,339707,"""Flushing Ave & Vanderbilt Ave""","""4762.05""","""Washington Ave & Park Ave""","""4724.03""",40.69795,-73.970776,40.696102,-73.96751,"""casual"""
"""0ACCA694D85B7BDE""","""classic_bike""",2022-12-28 11:24:17.103,2023-01-02 09:47:32.290,426195,"""E 138 St & Park Ave""","""7803.02""",null,null,40.812636,-73.92925,null,null,"""casual"""
"""4EDDEF85A6E6BF1B""","""classic_bike""",2022-12-28 11:27:39.038,2023-01-02 19:42:11.504,461672,"""West St & Chambers St""","""5329.03""","""11 Ave & W 27 St""","""6425.04""",40.717548,-74.013221,40.751396,-74.005226,"""casual"""


### 2.1. Filter Data

1. Remove any data that doesn't have an end station (probably bad data)
2. Remove any data that has a ride time of over 2 hours (probably bad data)

In [ ]:
# remove any start or end_station_name = null
df = df.filter(pl.col("end_station_name").is_not_null())
df= df.filter(pl.col("end_station_id").is_not_null())
df = df.filter(pl.col("start_station_name").is_not_null())
df= df.filter(pl.col("start_station_id").is_not_null())

# filter data out for ride_time < 2 hours or way too short ones like 1 minute
df = df.filter(pl.col("ride_time_seconds") < 2 * 3600)   
df = df.filter(pl.col("ride_time_seconds") >= 60)

df = df.filter(pl.col("start_time").dt.year() == 2023)
df = df.with_columns([
    pl.col("start_time").dt.month().alias("month"),
])

# filter data out for ride time that is negative or zero (not possible)
df = df.filter(pl.col("ride_time_seconds") > 0)

# Remove rides with identical start/end stations (probably some error)
df = df.filter(
    ~(
        (pl.col("start_station_id") == pl.col("end_station_id"))
    )
)

df = df.filter(
    ~(
        (pl.col("start_lat") == pl.col("end_lat"))
        & (pl.col("start_lng") == pl.col("end_lng"))
    )
)

# 3. Check Data

In [11]:
# Rideable type counts
df.select(pl.col("rideable_type").value_counts())

rideable_type
struct[2]
"{""electric_bike"",17416253}"
"{""classic_bike"",17372605}"


In [12]:
# Member type counts
df.select(pl.col("member_casual").value_counts())

member_casual
struct[2]
"{""casual"",6469110}"
"{""member"",28319748}"
